```{contents}
```
## Weight Initialization

### Why Weight Initialization Matters

Weight initialization determines how signals propagate through a neural network **before learning begins**.
A poor initialization can cause:

* **Vanishing gradients** → network fails to learn
* **Exploding gradients** → unstable training
* **Slow convergence** → inefficient optimization

The goal is to start with weights that preserve the **variance of activations and gradients** across layers.

---

### Core Intuition

For a layer with input $x$ and weights $W$:

$
y = W x
$

If weights are too large → activations explode
If too small → activations vanish

A good initializer keeps:

$
Var(\text{activations}) \approx Var(\text{gradients}) \approx \text{constant}
$

across depth.

---

### Main Initialization Families

| Method          | Designed for  | Core Idea             |
| --------------- | ------------- | --------------------- |
| Zero / Constant | none          | breaks symmetry       |
| Random Normal   | baseline      | unstable              |
| Xavier / Glorot | tanh, sigmoid | variance preservation |
| He / Kaiming    | ReLU family   | rectifier correction  |
| LeCun           | SELU          | self-normalization    |
| Orthogonal      | deep RNNs     | gradient stability    |


### Main Weight Initialization Techniques

| Method          | Best For         | Weight Distribution  | Key Property              |
| --------------- | ---------------- | -------------------- | ------------------------- |
| Zero            | None             | All zeros            | Breaks learning           |
| Random Normal   | Shallow nets     | 𝒩(0, σ²)            | Often unstable            |
| Xavier (Glorot) | Tanh / Sigmoid   | 𝒩(0, 1/fan_in)      | Preserves variance        |
| He (Kaiming)    | ReLU / variants  | 𝒩(0, 2/fan_in)      | Compensates ReLU sparsity |
| Orthogonal      | RNNs / deep nets | Orthonormal          | Preserves gradient norm   |
| LSUV            | Very deep nets   | Layer-wise rescaling | Auto variance fixing      |


### Initialization vs Activation Compatibility

| Activation  | Recommended Initialization |
| ----------- | -------------------------- |
| Sigmoid     | Xavier                     |
| Tanh        | Xavier                     |
| ReLU        | He                         |
| LeakyReLU   | He                         |
| SELU        | LeCun normal               |
| Transformer | Xavier + LayerNorm         |

---

### Advanced Variants

| Variant               | Purpose                             |
| --------------------- | ----------------------------------- |
| LSUV                  | Automatic variance normalization    |
| LeCun Normal          | Self-normalizing networks           |
| Scaled Initialization | Transformers                        |
| Fixup Init            | Residual nets without normalization |


---

### Xavier (Glorot) Initialization

Designed for **tanh / sigmoid** networks.

$
Var(W) = \frac{2}{n_{in} + n_{out}}
$

Keeps forward and backward variance balanced.

```python
nn.init.xavier_uniform_(layer.weight)
```

---

### He (Kaiming) Initialization

Optimized for **ReLU / LeakyReLU**.

$
Var(W) = \frac{2}{n_{in}}
$

Accounts for half activations being zeroed by ReLU.

```python
nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
```

---

### LeCun Initialization

For **SELU** activations (self-normalizing nets).

$
Var(W) = \frac{1}{n_{in}}
$

```python
nn.init.normal_(layer.weight, mean=0, std=1/math.sqrt(n_in))
```

---

### Orthogonal Initialization

Keeps transformations length-preserving; excellent for **deep nets & RNNs**.

```python
nn.init.orthogonal_(layer.weight)
```

---

### Complete PyTorch Demonstration

```python
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)

        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu")
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)
```

---

### Initialization Workflow in Practice

1. Choose activation functions
2. Select matching initializer
3. Initialize all learnable layers
4. Verify variance of activations
5. Begin training

---

### Failure Modes and Remediation

| Symptom             | Cause                    | Fix                        |
| ------------------- | ------------------------ | -------------------------- |
| Vanishing gradients | too small weights        | Xavier / He                |
| Exploding gradients | too large weights        | He + gradient clipping     |
| Slow convergence    | poor variance            | correct initializer        |
| Unstable training   | mismatch with activation | align init with activation |

---

### Advanced Variants

* **LSUV initialization** — layer-wise variance normalization
* **Scaled Orthogonal** — stabilizes extremely deep nets
* **Data-dependent initialization** — uses sample batch to tune scales

---

### Key Takeaways

* Initialization controls **trainability and stability**
* Must match **activation function**
* Modern defaults:

  * ReLU → **He initialization**
  * tanh/sigmoid → **Xavier initialization**
  * SELU → **LeCun initialization**

Proper initialization often determines whether a deep network trains at all.